In [ ]:
%load_ext autoreload
%load_ext sql
%autoreload 2

import sqlite3

import matplotlib.pyplot as plt
import pandas as pd
from config import settings

In [5]:
# from dotenv import load_dotenv
# import os

# load_dotenv()
# print(os.getenv("ALPHA_API_KEY"))



## Building a `get_daily` function that gets data from the AlphaVantage API and returns a clean DataFrame. 

- [What's a function?](../%40textbook/02-python-advanced.ipynb#Functions)
- [Write a function in Python.](../%40textbook/02-python-advanced.ipynb#Functions)

In [ ]:
# `get_daily` Function
def get_daily(ticker, output_size="full"): 
    """Get daily time series of an equity from AlphaVantage API.
    Parameters
    ----------
    ticker : str
        The ticker symbol of the equity.
    output_size : str, optional
        Number of observations to retrieve. "compact" returns the
        latest 100 observations. "full" returns all observations for
        equity. By default "full".

    Returns
    -------
    pd.DataFrame
        Columns are 'open', 'high', 'low', 'close', and 'volume'.
        All are numeric.
    """
    # Create URL (8.1.5)
    url = (
        "https://learn-api.wqu.edu/1/data-services/alpha-vantage/query?"
        "function=TIME_SERIES_DAILY&"
        f"symbol={ticker}&"
        f"outputsize={output_size}&"
        f"datatype=json&"
        f"apikey={settings.alpha_api_key}"
    )

    try:
        # Send request to API (8.1.6)
        response = requests.get(url, timeout=10)  # timeout avoids hanging forever
        response.raise_for_status()  # raises HTTPError for 4xx/5xx
        
        # Extract JSON data from response (8.1.10)
        response_data = response.json()
        
        # AlphaVantage sometimes returns an "Error Message" or "Note" instead of data
        if "Time Series (Daily)" not in response_data.keys():
            raise Exception(
                f"Invalid API call: Check that ticker symbol '{ticker}' is correct."
            )
        if "Error Message" in response_data:
            raise ValueError(f"API Error: {response_data['Error Message']}")
        if "Note" in response_data:  
            raise ValueError(f"API Note (probably rate-limited): {response_data['Note']}")
        
        # Parse time series data
        ts = response_data.get("Time Series (Daily)")
        if not ts:
            raise ValueError("Unexpected response format (missing 'Time Series (Daily)')")
    
        # Read data into DataFrame (8.1.12 & 8.1.13)
        stock_data = response_data["Time Series (Daily)"]
        df = pd.DataFrame.from_dict(stock_data, orient="index", dtype=float)
     
        # Convert index to `DatetimeIndex` named "date" (8.1.14)
        df.index = pd.to_datetime(df.index)
        # Name index "date"
        df.index.name = "date"
        
        # Remove numbering from columns (8.1.15)
        df.columns = [ c.split(". ")[1] for c in df.columns ]
    
        
        # Return DataFrame
        return df
        
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
        return pd.DataFrame()

    except ValueError as e:
        print(f"Data error: {e}")
        return pd.DataFrame()


**Assignments tasks**

In [ ]:
%load_ext autoreload
%autoreload 2

from arch.univariate.base import ARCHModelResult

In [ ]:
# Import your libraries here
import os
import sqlite3
from glob import glob

import joblib
import pandas as pd
import requests
from config import settings
from data import SQLRepository
from IPython.display import VimeoVideo

Create a URL to get all the stock data for MTN Group ("MTNOY") from AlphaVantage in JSON format. Be sure to use the https://learn-api.wqu.edu hostname. And don't worry: your submission won't include your API key!

In [ ]:
ticker = "MTNOY"
output_size = "full"
data_type = "json"

url = (
    "https://learn-api.wqu.edu/1/data-services/alpha-vantage/query?"
    "function=TIME_SERIES_DAILY&"
    f"symbol={ticker}&"
    f"outputsize={output_size}&"
    f"datatype={data_type}&"
    f"apikey={settings.alpha_api_key}"
)

response = requests.get(url=url)
response_code = response.status_code
response_text = response.text
response_data = response.json()
response_data.keys()


print("response_data type:", type(response_data))
print("response_text type:", type(response_text))
print(response_text[:200])
print("code type:", type(response_code))
response_code
print("response type:", type(response))
print("url type:", type(url))
url

In [ ]:
# Extract `"Time Series (Daily)"` value from `response_data`
stock_data = response_data["Time Series (Daily)"]

print("stock_data type:", type(stock_data))

# Extract data for one of the days in `stock_data`
stock_data["2025-08-22"]

Create a DataFrame df_mtnoy with all the stock data for MTN. Make sure that the DataFrame has the correct type of index and column names. The grader will evaluate your work by looking at the row in df_mtnoy for 6 December 2021.

In [ ]:
# If your index is messed up, reset it
df_mtnoy = pd.DataFrame.from_dict(stock_data, orient="index", dtype=float)
# Convert `df_mtnoy` index to `DatetimeIndex`
df_mtnoy.index = pd.to_datetime(df_mtnoy.index)

# Name index "date"
df_mtnoy.index.name = "date"

# Remove numbering from `df_ambuja` column names
df_mtnoy.columns = [ c.split(". ")[1] for c in df_mtnoy.columns ]

print("df_mtnoy type:", type(df_mtnoy))
df_mtnoy.head()

Connect to the database whose name is stored in the .env file for this project. Be sure to set the check_same_thread argument to False. Assign the connection to the variable connection. The grader will evaluate your work by looking at the database location assigned to connection.

In [ ]:
connection = sqlite3.connect(database=settings.db_name, check_same_thread=False)
# connection

# Insert `MTNOY` data into database
# Create instance of class
repo = SQLRepository(connection=connection)

Insert df_mtnoy into your database. The grader will evaluate your work by looking at the first five rows of the MTNOY table in the database.

In [ ]:
response = repo.insert_table(table_name="MTNOY", records=df_mtnoy, if_exists="replace")
response

Read the MTNOY table from your database and assign the output to df_mtnoy_read. The grader will evaluate your work by looking at the row for 27 April 2022.

In [ ]:
df_mtnoy_read = repo.read_table(table_name="MTNOY")

print("df_mtnoy_read type:", type(df_mtnoy_read))
print("df_mtnoy_read shape:", df_mtnoy_read.shape)
df_mtnoy_read.head()

Create a Series y_mtnoy with the 2,500 most recent returns for MTN. The grader will evaluate your work by looking at the volatility for 9 August 2022.

In [ ]:
def wrangle_data(ticker, n_observations):
    """Extract table data from database. Calculate returns.

    Parameters
    ----------
    ticker : str
        The ticker symbol of the stock (also table name in database).

    n_observations : int
        Number of observations to return.

    Returns
    -------
    pd.Series
        Name will be `"return"`. There will be no `NaN` values.
    """
    # Get table from database
    df = repo.read_table(table_name=ticker, limit=n_observations + 1)

    # Sort DataFrame ascending by date
    df.sort_index(ascending=True, inplace=True)

    # Create "return" column
    df["return"] = df["close"].pct_change() * 100

    # Return returns
    return df["return"].dropna()

In [ ]:
y_mtnoy = wrangle_data(ticker="MTNOY", n_observations=2500)

print("y_mtnoy type:", type(y_mtnoy))
print("y_mtnoy shape:", y_mtnoy.shape)
y_mtnoy.head()

Calculate daily volatility for y_mtnoy, and assign the result to mtnoy_daily_volatility.

In [ ]:
mtnoy_daily_volatility = y_mtnoy.std()

print("mtnoy_daily_volatility type:", type(mtnoy_daily_volatility))
print("MTN Daily Volatility:", mtnoy_daily_volatility)

In [ ]:
import numpy as np

mtnoy_annual_volatility = mtnoy_daily_volatility * np.sqrt(252)

print("mtnoy_annual_volatility type:", type(mtnoy_annual_volatility))
print("MTN Annual Volatility:", mtnoy_annual_volatility)

Create a time series line plot for y_mtnoy. Be sure to label the x-axis "Date", the y-axis "Returns", and use the title "Time Series of MTNOY Returns".

In [ ]:
import matplotlib.pyplot as plt

# Create `fig` and `ax`
fig, ax = plt.subplots(figsize=(15, 6))

# Plot `y_mtnoy` on `ax`
y_mtnoy.plot(ax=ax,)

# Add x-axis label
plt.xlabel("Date")
plt.ylabel("Returns")
# Add title
plt.title("Time Series of MTNOY Returns")

Create an ACF plot of the squared returns for MTN. Be sure to label the x-axis "Lag [days]", the y-axis "Correlation Coefficient", and use the title "ACF of MTNOY Squared Returns".

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
# Create `fig` and `ax`
fig, ax = plt.subplots(figsize=(15, 6))

# Create ACF of squared, standardized residuals
plot_acf(y_mtnoy**2, ax=ax)

# Add axis labels
plt.xlabel("Lag [days]")
plt.ylabel("Correlation Coefficient");

# Add title
plt.title("ACF of MTNOY Squared Returns")

In [ ]:
# Create `fig` and `ax`
fig, ax = plt.subplots(figsize=(15, 6))

# Create ACF of squared, standardized residuals
plot_pacf(y_mtnoy**2, ax=ax)

# Add axis labels
plt.xlabel("Lag [days]")
plt.ylabel("Correlation Coefficient");

# Add title
plt.title("PACF of MTNOY Squared Returns")

Create a training set y_mtnoy_train that contains the first 80% of the observations in y_mtnoy.

In [ ]:
cutoff_test = int(len(y_mtnoy) * 0.8)
y_mtnoy_train = y_mtnoy.iloc[:cutoff_test]

print("y_mtnoy_train type:", type(y_mtnoy_train))
print("y_mtnoy_train shape:", y_mtnoy_train.shape)
y_mtnoy_train.head()

Build and fit a GARCH model using the data in y_mtnoy. Try different values for p and q, using the summary to assess its performance. The grader will evaluate whether your model is the correct data type.

In [ ]:
from arch import arch_model

# Build and train model
model = arch_model(
    y=y_mtnoy,
    p=3,
    q=3,
    rescale=False
).fit(disp=0)
print("model type:", type(model))

# Show model summary
model.summary()

Plot the standardized residuals for your model. Be sure to label the x-axis "Date", the y-axis "Value", and use the title "MTNOY GARCH Model Standardized Residuals".

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(15, 6))

# Plot standardized residuals
model.std_resid.plot(ax=ax, label="Standardized Residuals")

# Add axis labels
plt.xlabel("Date")
plt.ylabel("Value")
plt.title("MTNOY GARCH Model Standardized Residuals")

In [ ]:
# Create `fig` and `ax`
fig, ax = plt.subplots(figsize=(15, 6))

# Create ACF of squared, standardized residuals
plot_acf(model.std_resid**2, ax=ax)

# Add axis labels
plt.xlabel("Lag [days]")
plt.ylabel("Correlation Coefficient")

# Add title
plt.title("ACF of MTNOY GARCH Model Standardized Residuals")

Create an ACF plot of the squared, standardized residuals of your model. Be sure to label the x-axis "Lag [days]", the y-axis "Correlation Coefficient", and use the title "ACF of MTNOY GARCH Model Standardized Residuals".

In [ ]:
# Create `fig` and `ax`
fig, ax = plt.subplots(figsize=(15, 6))

# Create ACF of squared, standardized residuals
plot_acf(model.std_resid**2, ax=ax)

# Add axis labels
plt.xlabel("Lag [days]")
plt.ylabel("Correlation Coefficient")

# Add title
plt.title("ACF of MTNOY GARCH Model Standardized Residuals")

Change the fit method of your GarchModel class so that, when a model is done training, two more attributes are added to the object: self.aic with the AIC for the model, and self.bic with the BIC for the model. When you're done, use the cell below to check your work.

In [ ]:
# Import `build_model` function
from main import build_model

# Build model using new `MTNOY` data
model = build_model(ticker="MTNOY", use_new_data=True)

# Wrangle `MTNOY` returns
model.wrangle_data(n_observations=2500)

# Fit GARCH(1,1) model to data
model.fit(p=1, q=1)

# Does model have AIC and BIC attributes?
assert hasattr(model, "aic")
assert hasattr(model, "bic")

Change the fit_model function in the main module so that the "message" it returns includes the AIC and BIC scores. For example, the message should look something like this:

In [ ]:
# Import `FitIn` class and `fit_model` function
from main import FitIn, fit_model

# Instantiate `FitIn` object
request = FitIn(ticker="MTNOY", use_new_data=False, n_observations=2500, p=1, q=1)

# Build model and fit to data, following parameters in `request`
fit_out = fit_model(request=request)

# Inspect `fit_out`
fit_out

Create a post request to hit the "/fit" path running at "http://localhost:8008". You should train a GARCH(1,1) model on 2500 observations of the MTN data you already downloaded. Pass in your parameters as a dictionary using the json argument. The grader will evaluate the JSON of your response.

Go to the command line, navigate to the directory for this project, and start your app server by entering the following command.
**uvicorn main:app --reload --workers 1 --host localhost --port 8008**


In [ ]:
# URL of `/fit` path
url = "http://localhost:8008/fit"
# Data to send to path
json = {
    "ticker": "MTNOY",
    "use_new_data": False,
    "n_observations": 2500,
    "p": 1,
    "q": 1
}
# Response of post request
response = requests.post(url=url, json=json)

print("response type:", type(response))
print("response status code:", response.status_code)

Create a post request to hit the "/predict" path running at "http://localhost:8008". You should get the 5-day volatility forecast for MTN. When you're satisfied, submit your work to the grader.

In [ ]:
# URL of `/predict` path
url = "http://localhost:8008/predict"
# Data to send to path
json = { "ticker": "MTNOY", "n_days": 5, "use_new_data": False }

# Response of post request
response = requests.post(url=url, json=json)

print("response type:", type(response))
print("response status code:", response.status_code)

**Task 8.3.20:** Complete the code below to do walk-forward validation on your `model`. Then run the following code block to visualize the model's test predictions.

- [What's walk-forward validation?](../%40textbook/17-ts-core.ipynb#Walk-Forward-Validation)
- [Perform walk-forward validation for time series model.](../%40textbook/17-ts-core.ipynb#Walk-Forward-Validation)

In [ ]:
# Create empty list to hold predictions
predictions = []

# Calculate size of test data (20%)
test_size = int(len(y_ambuja) * 0.2)

# Walk forward
for i in range(test_size):
    # Create test data
    y_train = y_ambuja.iloc[: -(test_size - i)]

    # Train model
    model = arch_model(y=y_train, p=1, q=1, rescale=False).fit(disp=0)

    # Generate next prediction (volatility, not variance)
    next_pred = model.forecast(horizon=1, reindex=False).variance.iloc[0,0] ** 0.5 #to get the volatility

    # Append prediction to list
    predictions.append(next_pred)

# Create Series from predictions list
y_test_wfv = pd.Series(predictions, index=y_ambuja.tail(test_size).index)

print("y_test_wfv type:", type(y_test_wfv))
print("y_test_wfv shape:", y_test_wfv.shape)
y_test_wfv.head()

**Task 8.3.15:** Create a time series plot with the Ambuja returns and the conditional volatility for your `model`. Be sure to include axis labels and add a legend.

- [Make a line plot with time series data in pandas.](../%40textbook/07-visualization-pandas.ipynb#Line-Plots)

In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))

# Plot `y_ambuja_train`
y_ambuja_train.plot(ax=ax, label="Ambuja Daily Returns")

# Plot conditional volatility * 2
(2 * model.conditional_volatility).plot(
    ax=ax, color="C1", label="2D Conditional Volatility", linewidth=3
)

# Plot conditional volatility * -2
(-2 * model.conditional_volatility.rename("")).plot(
    ax=ax, color="C1",  linewidth=3
)

# Add axis labels
plt.xlabel("Date")


# Add legend
plt.legend();

In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))

# Plot returns for test data
y_ambuja.tail(test_size).plot(ax=ax, label="Ambuja Return")

# Plot volatility predictions * 2
(2 * y_test_wfv).plot(ax=ax, c="C1", label="2 SD Predicted Volatility")

# Plot volatility predictions * -2
(-2 * y_test_wfv).plot(ax=ax, c="C1")

# Label axes
plt.xlabel("Date")
plt.ylabel("Return")

# Add legend
plt.legend();

In [ ]:
prediction = model.forecast(horizon=5, reindex=False).variance ** 0.5
start = prediction.index[0] + pd.DateOffset(days=1)
prediction_dates = pd.bdate_range(start=start, periods=prediction.shape[1])
prediction_index = [d.isoformat() for d in prediction_dates]
prediction_index

In [ ]:
# Generate 5-day volatility forecast
prediction = model.forecast(horizon=5, reindex=False).variance ** 0.5
print(prediction)

# Calculate forecast start date
start = prediction.index[0] + pd.DateOffset(days=1)

# Create date range
prediction_dates = pd.bdate_range(start=start, periods=prediction.shape[1])

# Create prediction index labels, ISO 8601 format
prediction_index = [d.isoformat() for d in prediction_dates]

print("prediction_index type:", type(prediction_index))
print("prediction_index len:", len(prediction_index))
prediction_index[:3]

In [ ]:
def clean_prediction(prediction):
    """Reformat model prediction to JSON.

    Parameters
    ----------
    prediction : pd.DataFrame
        Variance from a `ARCHModelForecast`

    Returns
    -------
    dict
        Forecast of volatility. Each key is date in ISO 8601 format.
        Each value is predicted volatility.
    """
    # Calculate forecast start date
    start = prediction.index[0] + pd.DateOffset(days=1)
    
    # Create date range
    prediction_dates = pd.bdate_range(start=start, periods=prediction.shape[1])
    
    # Create prediction index labels, ISO 8601 format
    prediction_index = [d.isoformat() for d in prediction_dates] 

    # Extract predictions from DataFrame, get square root
    data = prediction.values.flatten() ** 0.5 #to get the Volatility

    # Combine `data` and `prediction_index` into Series
    prediction_formatted = pd.Series(data, index=prediction_index)

    # Return Series as dictionary
    return prediction_formatted.to_dict()

In [ ]:
prediction = model.forecast(horizon=10, reindex=False).variance
prediction_formatted = clean_prediction(prediction)

# Is `prediction_formatted` a dictionary?
assert isinstance(prediction_formatted, dict)

# Are keys correct data type?
assert all(isinstance(k, str) for k in prediction_formatted.keys())

# Are values correct data type
assert all(isinstance(v, float) for v in prediction_formatted.values())

prediction_formatted

https://ds-lab.namespace.im/a/bd110c5d/launch/0b7dfb01-a1f0-4747-b1b3-bc0f5ba3b9c8?page=5